# 11.01 位图集合的数据结构设计

## 小节概述

本节先建立数据结构语义真值：集合的逻辑含义不因物理表示而改变。你将实现位图的下标映射、插入、删除、查询和集合交，并用 <code>U=31/32/33</code> 检查末字与物理填充。

开始前请先阅读 [11.00 章节介绍](./11.00_chapter_intro.ipynb)，并确认已进入带有 CANN 与昇腾 NPU 的 CANNLab 环境。

### 本节目标

完成 11.01 后，你应能够：

1. 比较稠密数组、哈希集合和位图的空间与访问特征；
2. 使用字下标与 bit 下标实现成员操作和集合交；
3. 区分逻辑全集、有效位图字和 32 Byte 对齐后的物理跨度；
4. 用末字掩码和物理填充验证 <code>U=31/32/33</code> 的边界。

<strong>建议用时：</strong> 35 分钟。<strong>本节产出：</strong> 可执行的位图集合参考模型、边界检查和物理存储量推导。

## 教程内容

In [ ]:
from pathlib import Path
import shutil
import subprocess
import sys

def locate_chapter():
    relative = Path('contrib/tutorials/data_structures_compute/11_bitmap_set_bandwidth')
    for candidate in [Path.cwd(), Path.cwd() / relative, *[p / relative for p in Path.cwd().parents]]:
        if (candidate / 'src' / 'demo' / 'bitmap_and.asc').is_file():
            return candidate.resolve()
    raise FileNotFoundError('未找到实验 11 目录，请从 cann-learning-hub 仓库内打开 Notebook。')

CHAPTER_DIR = locate_chapter()
print('chapter :', CHAPTER_DIR)
print('python  :', sys.version.split()[0])
print('cmake   :', shutil.which('cmake') or 'NOT FOUND')
npu = subprocess.run(['npu-smi', 'info'], text=True, capture_output=True, check=False)
print('npu-smi :', 'PASS' if npu.returncode == 0 else 'FAILED')

> <strong>检查点：</strong> 输出应包含实验目录、Python 与 CMake 路径，<code>npu-smi</code> 应为 <code>PASS</code>。如果环境检查失败，先修复环境，不要把构建错误误判成位图逻辑错误。

### 1. 从集合抽象到物理表示

设全集为 $\Omega=\{0,1,\ldots,U-1\}$，集合需要支持 <code>Add</code>、<code>Remove</code>、<code>Contains</code> 与 <code>Intersection</code>。同一个抽象接口可以由不同数据结构实现：

| 表示 | 空间量级 | 成员查询 | 批量集合交 | 典型条件 |
| --- | --- | --- | --- | --- |
| 稠密 <code>uint8</code> 数组 | $\Theta(U)$ Byte | 连续下标，$O(1)$ | 扫描 $U$ 个字节 | 全集有界、实现直接 |
| 哈希集合 | 约 $\Theta(|S|/\alpha)$ 个桶 | 平均 $O(1)$，地址离散 | 对较小集合逐元素查询 | 全集很大且集合稀疏 |
| <code>uint32</code> 位图 | $\Theta(U/8)$ Byte | 取字并测试一位，$O(1)$ | 扫描 $\lceil U/32\rceil$ 个字 | 全集有界、重复批量运算 |

位图并不总是优于哈希集合。本实验研究全集有界、集合需要在 NPU 上重复做批量交运算的场景。

### 2. 成员到位图的下标映射

一个 <code>uint32</code> 保存 32 个成员状态：

```text
wordIndex = value // 32
bitIndex  = value % 32
bitMask   = 1U << bitIndex

Add:      words[wordIndex] |= bitMask
Remove:   words[wordIndex] &= ~bitMask
Contains: (words[wordIndex] & bitMask) != 0
```

元素 35 位于第 1 个字的第 3 位。数组下标从 0 开始，所以第 2 个字表示逻辑元素 32～63。

### 3. 逻辑长度、有效字与物理跨度

![U=33 的位图物理布局](images/bitmap_layout.svg)

必须区分三个量：

```text
logicalSize   = U
validWords    = CeilDiv(U, 32)
physicalWords = AlignUp(validWords, 8)
```

<code>validWords</code> 决定集合语义；<code>physicalWords</code> 使每行达到 32 Byte 的整数倍。若 <code>U % 32 != 0</code>，最后一个有效字的无效高位必须为 0；<code>[validWords, physicalWords)</code> 的填充字也必须为 0。

In [ ]:
WORD_BITS = 32
WORDS_PER_ALIGN = 8
UINT32_MASK = (1 << WORD_BITS) - 1

def ceil_div(value, divisor):
    return (value + divisor - 1) // divisor

def align_up(value, alignment):
    return ceil_div(value, alignment) * alignment

def bitmap_layout(universe_size):
    if universe_size < 1:
        raise ValueError('universe_size must be positive')
    valid_words = ceil_div(universe_size, WORD_BITS)
    physical_words = align_up(valid_words, WORDS_PER_ALIGN)
    remain = universe_size % WORD_BITS
    tail_mask = UINT32_MASK if remain == 0 else (1 << remain) - 1
    return valid_words, physical_words, tail_mask

for size in [1, 31, 32, 33, 255, 256, 257]:
    valid, physical, mask = bitmap_layout(size)
    print(f'U={size:>3} valid={valid:>2} physical={physical:>2} tailMask=0x{mask:08X}')

### 4. 可执行的位图集合参考模型

下面的 Python 类是本章的语义真值。Kernel 只改变执行位置和并行方式，不改变集合操作的含义。

In [ ]:
class BitmapSet:
    def __init__(self, universe_size):
        self.universe_size = universe_size
        self.valid_words, self.physical_words, self.tail_mask = bitmap_layout(universe_size)
        self.words = [0] * self.physical_words

    def _position(self, value):
        if not 0 <= value < self.universe_size:
            raise IndexError(f'{value} is outside [0, {self.universe_size})')
        return value // WORD_BITS, value % WORD_BITS

    def add(self, value):
        word, bit = self._position(value)
        self.words[word] |= 1 << bit

    def remove(self, value):
        word, bit = self._position(value)
        self.words[word] &= ~(1 << bit) & UINT32_MASK

    def contains(self, value):
        word, bit = self._position(value)
        return bool(self.words[word] & (1 << bit))

    def intersection(self, other):
        if self.universe_size != other.universe_size:
            raise ValueError('universe sizes must match')
        result = BitmapSet(self.universe_size)
        for word in range(self.valid_words):
            result.words[word] = self.words[word] & other.words[word]
        result.words[self.valid_words - 1] &= self.tail_mask
        return result

    def members(self):
        return [value for value in range(self.universe_size) if self.contains(value)]

    def invariants_hold(self):
        invalid_high = self.words[self.valid_words - 1] & (~self.tail_mask & UINT32_MASK)
        return invalid_high == 0 and all(word == 0 for word in self.words[self.valid_words:])

In [ ]:
for size in [31, 32, 33]:
    left = BitmapSet(size)
    right = BitmapSet(size)
    for value in [0, 2, 30, 31, 32]:
        if value < size and value % 2 == 0:
            left.add(value)
        if value < size and value >= 2:
            right.add(value)
    both = left.intersection(right)
    expected = sorted(set(left.members()) & set(right.members()))
    print(f'U={size}:', both.members(), 'physical=', both.words)
    assert both.members() == expected
    assert both.invariants_hold()
print('BOUNDARY_CHECK PASS')

### 5. 压缩比必须按物理行计算

稠密表示每行向上补齐到 32 个 <code>uint8</code>；位图每行向上补齐到 8 个 <code>uint32</code>。短行的对齐填充会显著降低实际压缩比。

In [ ]:
def storage_bytes(batch, universe):
    dense = batch * align_up(universe, 32)
    valid_words, physical_words, _ = bitmap_layout(universe)
    bitmap = batch * physical_words * 4
    return dense, bitmap

for batch, universe in [(1, 31), (1, 32), (1, 33), (256, 4096), (32, 1_000_003)]:
    dense, bitmap = storage_bytes(batch, universe)
    print(f'N={batch:>3} U={universe:>8} dense={dense:>10} bitmap={bitmap:>10} ratio={dense/bitmap:>5.2f}')

## 课后练习

1. 解释 <code>U=32</code> 时末字掩码为什么不能是 0。
2. 解释 <code>U=33</code> 为什么有 2 个有效字却分配 8 个物理字。
3. 比较 <code>U=10^6, |S|=10</code> 与 <code>U=10^6, |S|=5\times10^5</code> 时位图和哈希集合的空间选择。
4. 说明逐字集合交为什么能展平成一维连续任务。

独立完成后再打开答案。

In [ ]:
SHOW_ANSWER = False
answer_file = CHAPTER_DIR / 'answer' / '11.01_bitmap_set_structure' / 'answers.md'
print(answer_file.read_text(encoding='utf-8') if SHOW_ANSWER else '将 SHOW_ANSWER 改为 True 后重新运行。')

## 本节小结

本节建立了位图集合的语义真值：下标映射决定成员位置，有效字决定逻辑内容，末字掩码和物理填充保证边界正确。下一节把逐字集合交映射为 Ascend C 的多核、UB 与队列执行过程。

继续学习 [11.02 集合交算子与带宽实验](./11.02_bitmap_and_operator.ipynb)。